# Understanding Embeddings: A Hands-On Guide

This notebook demystifies how text embeddings work. By the end, you'll understand:

1. **What embeddings actually are** (not just "vectors")
2. **How transformers create them** (step by step)
3. **Why similar text = similar vectors** (cosine similarity)
4. **Why different models give different results** (and why you can't mix them)

**Requirements:** None beyond free Colab. No GPU needed.

## Setup

Run this cell first. It installs required packages (takes ~1 minute).

In [1]:
# Install required packages
!pip install -q sentence-transformers umap-learn plotly pandas numpy

import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

Setup complete!


In [2]:
# Core imports
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import plotly.express as px
import plotly.graph_objects as go
from typing import List

print("Imports successful!")

Imports successful!


---

## Part 1: What Is An Embedding?

An embedding is a **list of numbers** that represents the "meaning" of text.

Let's see one:

In [3]:
# Load a small, fast embedding model
# This downloads ~90MB on first run
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create an embedding for a simple sentence
text = "The cat sat on the mat."
embedding = model.encode(text)

print(f"Input text: '{text}'")
print(f"\nEmbedding type: {type(embedding)}")
print(f"Embedding shape: {embedding.shape}")
print(f"\nFirst 10 values: {embedding[:10]}")
print(f"\nMin value: {embedding.min():.4f}")
print(f"Max value: {embedding.max():.4f}")
print(f"Mean value: {embedding.mean():.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Input text: 'The cat sat on the mat.'

Embedding type: <class 'numpy.ndarray'>
Embedding shape: (384,)

First 10 values: [ 0.13023719 -0.01577286 -0.0367167   0.05798646 -0.05979176  0.03305371
  0.03012399  0.02892724 -0.01860527  0.05529658]

Min value: -0.1659
Max value: 0.1497
Mean value: 0.0004


### What just happened?

The model converted `"The cat sat on the mat."` into **384 numbers**.

These numbers encode the "meaning" of the sentence in a way that:
- Similar sentences → similar numbers
- Different meanings → different numbers

**Key insight:** The 384 dimensions aren't human-interpretable. Dimension 47 doesn't mean "animal" or "furniture". The meaning is distributed across ALL dimensions together.

---

## Part 2: How Does the Model Create Embeddings?

Let's peek inside the process:

In [4]:
# Step 1: TOKENIZATION
# The model breaks text into "tokens" (subwords)

tokenizer = model.tokenizer

text = "The cat sat on the mat."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Step 1: TOKENIZATION")
print("="*50)
print(f"Original text: '{text}'")
print(f"\nTokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"\nVocabulary size: {tokenizer.vocab_size:,} possible tokens")

Step 1: TOKENIZATION
Original text: 'The cat sat on the mat.'

Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat', '.']
Token IDs: [101, 1996, 4937, 2938, 2006, 1996, 13523, 1012, 102]

Vocabulary size: 30,522 possible tokens


In [5]:
# Let's see how different words get tokenized

test_words = [
    "cat",
    "cats", 
    "caterpillar",
    "embeddings",
    "supercalifragilisticexpialidocious"
]

print("How words become tokens:")
print("="*50)
for word in test_words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:40} → {tokens}")

How words become tokens:
cat                                      → ['cat']
cats                                     → ['cats']
caterpillar                              → ['cater', '##pi', '##llar']
embeddings                               → ['em', '##bed', '##ding', '##s']
supercalifragilisticexpialidocious       → ['super', '##cal', '##if', '##rag', '##ilis', '##tic', '##ex', '##pia', '##lid', '##oc', '##ious']


### Why subword tokenization?

- `"cat"` is common → single token
- `"caterpillar"` is rarer → split into `["cat", "##er", "##pillar"]`
- This lets the model handle words it's never seen by combining known pieces

The `##` prefix means "continuation of previous token".

In [6]:
# Step 2: TOKEN EMBEDDINGS
# Each token ID maps to a learned vector

import torch

# Get the raw token embeddings from the model
word_embeddings = model[0].auto_model.embeddings.word_embeddings

print("Step 2: TOKEN EMBEDDINGS")
print("="*50)
print(f"Embedding table shape: {word_embeddings.weight.shape}")
print(f"  → {word_embeddings.weight.shape[0]:,} tokens")
print(f"  → {word_embeddings.weight.shape[1]} dimensions each")

# Look up the embedding for "cat" (token ID varies by model)
cat_token_id = tokenizer.encode("cat")[1]  # Skip [CLS] token
cat_token_embedding = word_embeddings.weight[cat_token_id].detach().numpy()

print(f"\n'cat' token ID: {cat_token_id}")
print(f"'cat' token embedding (first 10 dims): {cat_token_embedding[:10]}")

Step 2: TOKEN EMBEDDINGS
Embedding table shape: torch.Size([30522, 384])
  → 30,522 tokens
  → 384 dimensions each

'cat' token ID: 4937
'cat' token embedding (first 10 dims): [ 0.03254607 -0.03286388  0.06211727  0.03036976 -0.09235375  0.0098366
  0.02301718 -0.04831578 -0.03698478  0.02337898]


In [7]:
# Step 3: TRANSFORMER LAYERS (the "magic")
#
# This is where context matters. The word "bank" gets different
# representations in "river bank" vs "bank account".

print("Step 3: TRANSFORMER LAYERS")
print("="*50)

# Count the layers
encoder = model[0].auto_model.encoder
num_layers = len(encoder.layer)
print(f"Number of transformer layers: {num_layers}")

# Each layer has:
print(f"\nEach layer contains:")
print(f"  - Self-attention (tokens 'look at' each other)")
print(f"  - Feed-forward network (process the combined info)")
print(f"  - Layer normalization (keep values stable)")

Step 3: TRANSFORMER LAYERS
Number of transformer layers: 6

Each layer contains:
  - Self-attention (tokens 'look at' each other)
  - Feed-forward network (process the combined info)
  - Layer normalization (keep values stable)


In [8]:
# Step 4: POOLING
# After all layers, we have one vector PER TOKEN.
# Pooling combines them into ONE vector for the whole sentence.

print("Step 4: POOLING")
print("="*50)

text = "The cat sat on the mat."
tokens = tokenizer.tokenize(text)

print(f"Text: '{text}'")
print(f"Tokens: {tokens}")
print(f"Number of tokens: {len(tokens)}")
print(f"\nAfter transformer: {len(tokens)} vectors of size 384")
print(f"After pooling: 1 vector of size 384")
print(f"\nPooling method: Mean (average all token vectors)")

Step 4: POOLING
Text: 'The cat sat on the mat.'
Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat', '.']
Number of tokens: 7

After transformer: 7 vectors of size 384
After pooling: 1 vector of size 384

Pooling method: Mean (average all token vectors)


### Summary: Text → Embedding Pipeline

```
"The cat sat on the mat."
           ↓
    [Tokenization]
           ↓
[the, cat, sat, on, the, mat, .]
           ↓
  [Token Embeddings]
           ↓
   7 vectors (384-dim each)
           ↓
 [6 Transformer Layers]
 (tokens exchange info)
           ↓
   7 context-aware vectors
           ↓
      [Mean Pooling]
           ↓
   1 vector (384-dim)
```

---

## Part 3: Cosine Similarity - How We Measure "Sameness"

Now that we have vectors, how do we know if two are similar?

In [9]:
# Cosine similarity measures the ANGLE between vectors
# (ignoring their length)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Calculate cosine similarity between two vectors."""
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

# Let's test with simple 2D vectors first
print("Cosine Similarity with Simple 2D Vectors")
print("="*50)

v1 = np.array([1, 0])  # Points right →
v2 = np.array([1, 0])  # Points right →
v3 = np.array([0, 1])  # Points up ↑
v4 = np.array([-1, 0]) # Points left ←

print(f"v1 = [1, 0] (pointing right)")
print(f"v2 = [1, 0] (pointing right)")
print(f"v3 = [0, 1] (pointing up)")
print(f"v4 = [-1, 0] (pointing left)")
print()
print(f"similarity(v1, v2) = {cosine_similarity(v1, v2):.2f}  (identical direction)")
print(f"similarity(v1, v3) = {cosine_similarity(v1, v3):.2f}  (perpendicular)")
print(f"similarity(v1, v4) = {cosine_similarity(v1, v4):.2f}  (opposite direction)")

Cosine Similarity with Simple 2D Vectors
v1 = [1, 0] (pointing right)
v2 = [1, 0] (pointing right)
v3 = [0, 1] (pointing up)
v4 = [-1, 0] (pointing left)

similarity(v1, v2) = 1.00  (identical direction)
similarity(v1, v3) = 0.00  (perpendicular)
similarity(v1, v4) = -1.00  (opposite direction)


In [11]:
# Visualize this in 2D

fig = go.Figure()

vectors = {
    'v1 [1,0]': ([0, 1], [0, 0]),
    'v2 [1,0]': ([0, 0.95], [0, 0.05]),  # Slightly offset to show overlap
    'v3 [0,1]': ([0, 0], [0, 1]),
    'v4 [-1,0]': ([0, -1], [0, 0]),
}

colors = ['blue', 'lightblue', 'green', 'red']

for (name, (x, y)), color in zip(vectors.items(), colors):
    fig.add_trace(go.Scatter(
        x=x, y=y, mode='lines+markers',
        name=name, line=dict(width=3, color=color),
        marker=dict(size=10)
    ))

fig.update_layout(
    title='2D Vectors and Their Directions',
    xaxis=dict(range=[-1.5, 1.5], title='X'),
    yaxis=dict(range=[-0.5, 1.5], title='Y', scaleanchor='x'),
    height=400, width=600
)

fig.show()

In [ ]:
# Now let's see cosine similarity with REAL embeddings

sentences = [
    "The cat sat on the mat.",
    "A kitten rested on the rug.",  # Similar meaning
    "The dog ran in the park.",      # Different but related (animals)
    "Python is a programming language.",  # Completely different
    "Machine learning uses neural networks.",  # Tech domain
]

# Get embeddings
embeddings = model.encode(sentences)

# Calculate similarity matrix
print("Cosine Similarity Matrix (Real Embeddings)")
print("="*60)
print()

# Create similarity matrix
n = len(sentences)
sim_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = cosine_similarity(embeddings[i], embeddings[j])

# Print with labels
short_labels = [s[:30] + "..." if len(s) > 30 else s for s in sentences]

df = pd.DataFrame(sim_matrix, 
                  index=[f"{i}: {l}" for i, l in enumerate(short_labels)],
                  columns=[str(i) for i in range(n)])
print(df.round(3).to_string())

print()
print("For search, you'd typically set a threshold like 0.3 or 0.4 - anything above that is relevant enough to return.")

Cosine Similarity Matrix (Real Embeddings)

                                          0      1      2      3      4
0: The cat sat on the mat.            1.000  0.612  0.067  0.020 -0.065
1: A kitten rested on the rug.        0.612  1.000  0.071  0.036 -0.067
2: The dog ran in the park.           0.067  0.071  1.000  0.041  0.004
3: Python is a programming langua...  0.020  0.036  0.041  1.000  0.255
4: Machine learning uses neural n... -0.065 -0.067  0.004  0.255  1.000
For search, you'd typically set a threshold like 0.3 or 0.4 - anything above that is relevant enough to return.


In [15]:
# Visualize as heatmap

fig = px.imshow(
    sim_matrix,
    labels=dict(color="Similarity"),
    x=[f"{i}" for i in range(n)],
    y=[s[:35] + "..." if len(s) > 35 else s for s in sentences],
    color_continuous_scale="RdYlGn",
    zmin=0, zmax=1
)

fig.update_layout(
    title="Semantic Similarity Heatmap",
    height=400,
    width=700
)

# Add text annotations
for i in range(n):
    for j in range(n):
        fig.add_annotation(
            x=j, y=i,
            text=f"{sim_matrix[i][j]:.2f}",
            showarrow=False,
            font=dict(color="black" if sim_matrix[i][j] > 0.5 else "white")
        )

fig.show()

### What do you notice?

- "cat sat on mat" ↔ "kitten rested on rug" = **high similarity** (~0.7-0.8)
  - Different words, same meaning!
  
- "cat sat on mat" ↔ "dog ran in park" = **medium similarity** (~0.4-0.5)
  - Both about animals and locations
  
- "cat sat on mat" ↔ "Python is a programming language" = **low similarity** (~0.1-0.2)
  - Completely different topics

**This is semantic search!** Find documents with high similarity to a query.

---

## Part 4: Visualizing Embeddings in 2D

384 dimensions is impossible to visualize. UMAP compresses them to 2D while preserving relationships.

In [ ]:
# More diverse sentences for interesting visualization

sentences = [
    # Animals
    "The cat sleeps on the couch.",
    "A kitten plays with yarn.",
    "The dog runs in the yard.",
    "A puppy chases its tail.",
    
    # Food
    "I love eating pizza for dinner.",
    "Spaghetti with meatballs is delicious.",
    "The chef prepared a gourmet meal.",
    "Breakfast includes eggs and toast.",
    
    # Technology
    "Python is a popular programming language.",
    "Machine learning requires large datasets.",
    "The neural network trained for hours.",
    "Software engineers write code daily.",
    
    # Nature
    "The mountains are covered in snow.",
    "Ocean waves crash on the beach.",
    "The forest is full of tall trees.",
    "Rivers flow from mountains to sea.",
]

categories = (['Animals'] * 4 + ['Food'] * 4 + 
              ['Technology'] * 4 + ['Nature'] * 4)

# Get embeddings
embeddings = model.encode(sentences)

print(f"Created {len(embeddings)} embeddings of dimension {embeddings.shape[1]}")

In [ ]:
# Use UMAP to reduce to 2D
import umap

reducer = umap.UMAP(
    n_neighbors=5,      # How many neighbors to consider
    min_dist=0.3,       # How tightly to pack points
    metric='cosine',    # Use cosine similarity
    random_state=42     # For reproducibility
)

embeddings_2d = reducer.fit_transform(embeddings)

print(f"Reduced from {embeddings.shape[1]} dimensions to {embeddings_2d.shape[1]}")

In [ ]:
# Visualize!

df = pd.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'text': sentences,
    'category': categories
})

fig = px.scatter(
    df, x='x', y='y', 
    color='category',
    hover_data=['text'],
    title='Embeddings Visualized in 2D (UMAP)',
    color_discrete_sequence=px.colors.qualitative.Set1
)

fig.update_traces(marker=dict(size=12))
fig.update_layout(height=600, width=800)

fig.show()

### What do you see?

Similar topics cluster together! Even though the model never saw these labels:
- Animal sentences are near each other
- Food sentences cluster together
- Tech sentences form their own group
- Nature sentences are grouped

**This is the power of embeddings** - they capture semantic meaning, not just keywords.

---

## Part 5: Why Different Models Give Different Embeddings

Let's compare two different embedding models.

In [ ]:
# Load a second model (different architecture/training)
model_mini = model  # Already loaded: all-MiniLM-L6-v2 (384 dim)
model_mpnet = SentenceTransformer('all-mpnet-base-v2')  # 768 dim, larger

print("Model Comparison:")
print("="*50)
print(f"Model 1: all-MiniLM-L6-v2")
print(f"  - Dimensions: 384")
print(f"  - Layers: 6")
print(f"  - Parameters: ~22M")
print(f"\nModel 2: all-mpnet-base-v2")
print(f"  - Dimensions: 768")
print(f"  - Layers: 12")
print(f"  - Parameters: ~110M")

In [ ]:
# Same text, different embeddings

text = "Machine learning is fascinating."

emb_mini = model_mini.encode(text)
emb_mpnet = model_mpnet.encode(text)

print(f"Text: '{text}'")
print(f"\nMiniLM embedding (first 10 of {len(emb_mini)}):")
print(f"  {emb_mini[:10].round(4)}")
print(f"\nMPNet embedding (first 10 of {len(emb_mpnet)}):")
print(f"  {emb_mpnet[:10].round(4)}")
print(f"\nNotice: Completely different numbers!")
print(f"         Different dimensions too ({len(emb_mini)} vs {len(emb_mpnet)})")

In [ ]:
# Can we compare embeddings from different models?

print("Can we compute similarity between different models?")
print("="*50)

try:
    # This will fail!
    sim = cosine_similarity(emb_mini, emb_mpnet)
    print(f"Similarity: {sim}")
except ValueError as e:
    print(f"ERROR: {e}")
    print(f"\nWhy? The vectors have different dimensions:")
    print(f"  - MiniLM: {len(emb_mini)} dimensions")
    print(f"  - MPNet: {len(emb_mpnet)} dimensions")
    print(f"\nYou CANNOT compare vectors from different models!")

In [ ]:
# But WITHIN each model, similarities are consistent

texts = [
    "I love machine learning.",
    "Deep learning is amazing.",
    "I enjoy cooking pasta."
]

emb_mini = model_mini.encode(texts)
emb_mpnet = model_mpnet.encode(texts)

print("Similarity Rankings (should be same order)")
print("="*50)
print(f"\nTexts:")
for i, t in enumerate(texts):
    print(f"  {i}: {t}")

print(f"\nSimilarity between 0 and 1 (both about ML):")
print(f"  MiniLM: {cosine_similarity(emb_mini[0], emb_mini[1]):.4f}")
print(f"  MPNet:  {cosine_similarity(emb_mpnet[0], emb_mpnet[1]):.4f}")

print(f"\nSimilarity between 0 and 2 (ML vs cooking):")
print(f"  MiniLM: {cosine_similarity(emb_mini[0], emb_mini[2]):.4f}")
print(f"  MPNet:  {cosine_similarity(emb_mpnet[0], emb_mpnet[2]):.4f}")

print(f"\nBoth models agree: 0-1 are more similar than 0-2!")

### Key Takeaway

**You cannot mix embeddings from different models!**

In Greg, this is why you have:
- `embedding_local` column (384 dim, local model)
- `embedding_openai` column (1536 dim, OpenAI)
- `embedding_provider` field to track which was used

If you index with Model A, you MUST query with Model A.

---

## Part 6: Semantic Search Demo

Let's build a tiny search engine to see this in action.

In [ ]:
# Our "document" corpus
documents = [
    "How to train a neural network from scratch",
    "Best practices for REST API design",
    "Introduction to machine learning algorithms",
    "Setting up PostgreSQL with Python",
    "Deep learning for natural language processing",
    "Building web applications with FastAPI",
    "Understanding transformer architectures",
    "Database optimization techniques",
    "Python async programming patterns",
    "Deploying ML models to production",
]

# Index them (create embeddings)
doc_embeddings = model.encode(documents)

print(f"Indexed {len(documents)} documents")
print(f"Each document → {doc_embeddings.shape[1]} dimensional vector")

In [ ]:
def search(query: str, top_k: int = 3) -> List[tuple]:
    """Search documents by semantic similarity."""
    # Embed the query
    query_embedding = model.encode(query)
    
    # Calculate similarity to all documents
    similarities = [
        cosine_similarity(query_embedding, doc_emb)
        for doc_emb in doc_embeddings
    ]
    
    # Sort by similarity (descending)
    results = sorted(
        zip(documents, similarities),
        key=lambda x: x[1],
        reverse=True
    )
    
    return results[:top_k]


# Test some queries
queries = [
    "How do I build AI models?",
    "database performance",
    "web development framework",
    "GPT and BERT explained",
]

for query in queries:
    print(f"\n🔍 Query: '{query}'")
    print("-" * 50)
    results = search(query)
    for doc, score in results:
        print(f"  [{score:.3f}] {doc}")

### Notice:

- "How do I build AI models?" → finds neural network and ML docs
- "database performance" → finds PostgreSQL and optimization docs
- "GPT and BERT explained" → finds transformer architecture doc

**No keyword matching!** The query "GPT and BERT" found "transformer architectures" even though those words don't appear in the document.

---

## Summary: What You Learned

1. **Embeddings** are lists of numbers (384-1536) that encode meaning

2. **The pipeline**: Text → Tokens → Token Embeddings → Transformer → Pooling → Final Vector

3. **Cosine similarity** measures how much two vectors point in the same direction (0-1)

4. **Different models = different spaces**: You cannot compare embeddings from different models

5. **Semantic search**: Query embedding + cosine similarity finds relevant documents without keyword matching

---

## Next Steps

Now that you understand embeddings:

1. **Experiment**: Change the sentences and see how similarities change
2. **Compare models**: Try `all-MiniLM-L6-v2` vs `all-mpnet-base-v2` on your own data
3. **Evaluate**: Which model works better for YOUR documents?
4. **Apply to Greg**: Use this understanding to debug embedding issues